# Base Model Evaluation Using Azure AI Foundry

Evaluate base models **directly** using Foundry's evaluation API:
- Same dataset as Phase 2 (rft_next_val_v2.jsonl - 62 scenarios)
- Same grader (zava_grader_response.py)
- System prompt + tools passed to base models
- **No agent deployment required!**

This allows us to:
- Compare base model capability vs agent performance
- Test models without deploying agents
- Understand pure model reasoning
- Identify best base models for the task

In [ ]:
import os
import json
import time
from datetime import datetime
from pathlib import Path

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.evaluation import evaluate

# Load environment
env_path = Path('../.env')
if env_path.exists():
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                key, value = line.split('=', 1)
                key = key.strip()
                value = value.split('#')[0].strip()
                if value:
                    os.environ[key] = value

PROJECT_ENDPOINT = os.environ.get('PROJECT_ENDPOINT')
TOOL_URL = os.environ.get('TOOL_URL')

print(f"✅ Environment loaded")
print(f"   Project: {PROJECT_ENDPOINT[:80]}...")
print(f"   Tool URL: {TOOL_URL}")

In [ ]:
# Connect to Azure AI Foundry
credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

print("✅ Connected to Azure AI Foundry")

In [ ]:
# Load system prompt
with open('../base-model-eval/system_prompt.md') as f:
    SYSTEM_PROMPT = f.read()

print(f"✅ Loaded system prompt ({len(SYSTEM_PROMPT)} characters)")

In [ ]:
# Configure tools for base model
# These will be passed to the model in each evaluation run
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_order_details",
            "description": "Retrieve complete order information including items, prices, dates, and customer details",
            "parameters": {
                "type": "object",
                "properties": {"order_id": {"type": "string", "description": "Order ID (e.g., ORD-001)"}},
                "required": ["order_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_fulfillment_status",
            "description": "Check shipping and delivery status for an order",
            "parameters": {
                "type": "object",
                "properties": {"order_id": {"type": "string", "description": "Order ID"}},
                "required": ["order_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_resolution_policy",
            "description": "Verify what resolutions are allowed for an order/item",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string"},
                    "line_item_id": {"type": "string"}
                },
                "required": ["order_id", "line_item_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_inventory",
            "description": "Check product availability for exchange",
            "parameters": {
                "type": "object",
                "properties": {"line_item_id": {"type": "string"}},
                "required": ["line_item_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_resolution",
            "description": "Calculate refund/credit amounts including restocking fees",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string"},
                    "line_item_id": {"type": "string"},
                    "resolution_type": {"type": "string", "enum": ["refund", "store_credit", "exchange"]}
                },
                "required": ["order_id", "line_item_id", "resolution_type"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "submit_resolution",
            "description": "Execute the approved resolution",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string"},
                    "line_item_id": {"type": "string"},
                    "action": {"type": "string", "enum": ["refund", "store_credit", "exchange", "cancel", "deny"]},
                    "amount": {"type": "number"},
                    "reason": {"type": "string"}
                },
                "required": ["order_id", "line_item_id", "action"]
            }
        }
    }
]

print(f"✅ Configured {len(TOOLS)} tools")

In [ ]:
# Load validation dataset (same as Phase 2)
dataset_path = '../data/rft_next_val_v2.jsonl'
print(f"✅ Dataset: {dataset_path}")

In [ ]:
# Load grader (same as Phase 2)
with open('../eval/zava_grader_response.py') as f:
    GRADER_SOURCE = f.read()

print(f"✅ Loaded grader ({len(GRADER_SOURCE)} characters)")
print(f"   File: eval/zava_grader_response.py")

In [ ]:
# Models to evaluate
MODELS = [
    "o4-mini-2025-04-16",
    "gpt-4-1",
    "gpt-4-1-mini",
    "gpt-4-1-nano",
    "gpt-5-4",
    "gpt-5-4-mini",
]

print(f"📊 Models to evaluate: {len(MODELS)}")
for i, model in enumerate(MODELS, 1):
    print(f"   {i}. {model}")

In [ ]:
# Submit base model evaluation to Foundry
from azure.ai.evaluation import EvaluationClient

eval_client = project_client.evaluation

# Configuration for base model evaluation
eval_name = f"base-model-eval-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"🚀 Submitting base model evaluation...")
print(f"   Name: {eval_name}")
print(f"   Models: {len(MODELS)}")
print(f"   Dataset: {dataset_path}")
print(f"\nNOTE: Check Foundry documentation for exact API.")
print(f"      This is a template - adjust based on actual Foundry API.")

# TEMPLATE - Adjust based on actual Foundry API:
# evaluation_result = eval_client.create(
#     name=eval_name,
#     models=MODELS,  # List of base model IDs
#     dataset=dataset_path,
#     system_prompt=SYSTEM_PROMPT,
#     tools=TOOLS,
#     grader={
#         "type": "python",
#         "source": GRADER_SOURCE
#     }
# )

print(f"\n⚠️  TODO: Update this cell with actual Foundry base model eval API")

## Monitor Evaluation

Once submitted, monitor the evaluation progress:
```python
# Get evaluation status
status = eval_client.get(evaluation_id)
print(f"Status: {status.state}")

# Get results when complete
results = eval_client.get_results(evaluation_id)
```